# Problem Formulation: Impact of Social Media on Student Life

### **Dataset Source**
* **Source:** Kaggle Dataset - [Impact of Social Media on Life](https://www.kaggle.com/datasets/harishyadav0506/impact-of-social-media-on-life) by Harish Yadav.
* **Data Overview:** The dataset contains 4,500 records surveying student demographics, social media behaviors (usage hours, platform preferences, late-night patterns), sleep details, academic metrics (GPA), and self-reported mental health indices.

---

### **Problem Statement**
Can we predict the **Overall Impact** (`Beneficial`, `Neutral`, or `Negative`) of social media on a student's life based on their demographics, usage behavior, sleep patterns, and mental health indices?

* **Task Type:** Multi-class Classification
* **Target Variable:** `Overall_Impact` (The overall perceived effect of social media usage: `Beneficial`, `Neutral`, or `Negative`)

---

### **Data Dictionary (Feature Definitions)**

| Feature Name | Data Type | Description |
| :--- | :--- | :--- |
| `Student_ID` | Object / Categorical | Unique identifier for each surveyed student. |
| `Age` | Integer | The age of the student in years. |
| `Gender` | Object / Categorical | The gender of the student (e.g., Male, Female). |
| `Academic_Level` | Object / Categorical | The education level of the student (e.g., High School, Undergraduate, Postgraduate). |
| `Primary_Platform` | Object / Categorical | The main social media app used by the student (e.g., Instagram, TikTok, Snapchat, YouTube, Reddit, LinkedIn). |
| `Daily_Usage_Hours` | Float | Average hours spent on social media per day. |
| `Weekend_Extra_Hours` | Float | Additional hours spent on social media during weekends. |
| `Device_Type` | Object / Categorical | Primary device used to access social media (e.g., Smartphone, Tablet, Laptop/PC). |
| `Sleep_Duration_Hours` | Float | Average hours of sleep obtained per night. |
| `Sleep_Quality_Score` | Integer | Self-reported rating of sleep quality (typically on a scale of 1 to 5). |
| `Late_Night_Usage` | Boolean | Indicates whether the student regularly uses social media late at night (True/False). |
| `Social_Comparison_Frequency` | Object / Categorical | How often the student compares themselves to others on social media (e.g., Always, Frequently, Sometimes, Rarely, Never). |
| `Perceived_Stress_Score` | Float | A psychological score indicating perceived stress level (ranging from 0 to 40). |
| `Mental_Health_Index` | Integer | A calculated index reflecting overall self-reported mental health status. |
| `Academic_Performance_GPA` | Float | The student's current Grade Point Average (GPA). |
| `Overall_Impact` | Object / Categorical | **[TARGET]** The perceived overall impact of social media on the student's life (`Beneficial`, `Neutral`, or `Negative`). |

### 1. Importing Libraries and Modules

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

import kagglehub
from kagglehub import KaggleDatasetAdapter

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE

# Evaluation Metrics & Explainability
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.inspection import permutation_importance

# Selected Classification Models
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

### 2. Dataset Loading & Overview

In [ ]:
file_path = "Social_media_impact_on_life.csv"

# Load the latest version using kagglehub
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "harishyadav0506/impact-of-social-media-on-life",
    file_path
)

print("Dataset Shape:", df.shape)
display(df.head(10))

In [ ]:
df.info()
display(df.describe())

### 3. Exploratory Data Analysis (EDA)

In [ ]:
sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Target variable distribution
sns.countplot(ax=axes[0], data=df, x='Overall_Impact', palette='viridis', hue='Overall_Impact', legend=False)
axes[0].set_title("Distribution of Target Variable (Overall Impact)")
axes[0].set_xlabel("Overall Impact")
axes[0].set_ylabel("Count")

# Daily Usage Hours vs Overall Impact
sns.boxplot(ax=axes[1], data=df, x='Overall_Impact', y='Daily_Usage_Hours', palette='muted', hue='Overall_Impact', legend=False)
axes[1].set_title("Daily Social Media Usage by Overall Impact")
axes[1].set_xlabel("Overall Impact")
axes[1].set_ylabel("Daily Usage Hours")

# Mental Health Index vs Overall Impact
sns.boxplot(ax=axes[2], data=df, x='Overall_Impact', y='Mental_Health_Index', palette='muted', hue='Overall_Impact', legend=False)
axes[2].set_title("Mental Health Index by Overall Impact")
axes[2].set_xlabel("Overall Impact")
axes[2].set_ylabel("Mental Health Index")

plt.tight_layout()
plt.show()

print("\nTarget Class Proportions:")
print(df['Overall_Impact'].value_counts(normalize=True))

### 4. Data Splitting (60% Train / 20% Validation / 20% Test)
Splitting the data before any imputation, encoding, or scaling prevents data leakage. We also drop `Student_ID` as it is a unique identifier and does not carry predictive signal.

In [ ]:
My_data = df.copy()

X = My_data.drop(columns=['Overall_Impact', 'Student_ID'])
y = My_data['Overall_Impact']

# 80% temp, 20% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 60% train (0.75 * 0.80), 20% validation (0.25 * 0.80)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print(f"X_train shape: {X_train.shape} (60%)")
print(f"X_val shape:   {X_val.shape} (20%)")
print(f"X_test shape:  {X_test.shape} (20%)")

### 5. Handling Missing Values & Duplicates
We inspect missing values across all splits, check outliers in columns containing missing values (`Perceived_Stress_Score` and `Academic_Performance_GPA`), and apply `SimpleImputer(strategy='median')`.

In [ ]:
print("Missing values in X_train:\n", X_train.isna().sum()[X_train.isna().sum() > 0])
print("\nMissing values in X_val:\n", X_val.isna().sum()[X_val.isna().sum() > 0])
print("\nMissing values in X_test:\n", X_test.isna().sum()[X_test.isna().sum() > 0])
print("\nDuplicate rows in X_train:", X_train.duplicated().sum())

# Visualize outliers in columns with missing values
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.boxplot(X_train["Perceived_Stress_Score"].dropna())
plt.title("Distribution of Perceived Stress Score")
plt.ylabel("Stress Score")
plt.xticks([])

plt.subplot(1, 2, 2)
plt.boxplot(X_train["Academic_Performance_GPA"].dropna())
plt.title("Distribution of Academic Performance GPA")
plt.ylabel("GPA")
plt.xticks([])

plt.tight_layout()
plt.show()

# Median imputation fit strictly on X_train
stress_imputer = SimpleImputer(strategy='median')
gpa_imputer = SimpleImputer(strategy='median')

X_train[['Perceived_Stress_Score']] = stress_imputer.fit_transform(X_train[['Perceived_Stress_Score']])
X_val[['Perceived_Stress_Score']] = stress_imputer.transform(X_val[['Perceived_Stress_Score']])
X_test[['Perceived_Stress_Score']] = stress_imputer.transform(X_test[['Perceived_Stress_Score']])

X_train[['Academic_Performance_GPA']] = gpa_imputer.fit_transform(X_train[['Academic_Performance_GPA']])
X_val[['Academic_Performance_GPA']] = gpa_imputer.transform(X_val[['Academic_Performance_GPA']])
X_test[['Academic_Performance_GPA']] = gpa_imputer.transform(X_test[['Academic_Performance_GPA']])

print("Remaining missing values in Train:", X_train.isna().sum().sum())
print("Remaining missing values in Val:  ", X_val.isna().sum().sum())
print("Remaining missing values in Test: ", X_test.isna().sum().sum())

### 6. Feature Encoding, Target Encoding & Scaling
1. **Ordinal Encoding:** `Social_Comparison_Frequency` and `Academic_Level` follow a natural hierarchy.
2. **One-Hot Encoding:** Nominal features (`Gender`, `Primary_Platform`, `Device_Type`) are encoded using `OneHotEncoder(handle_unknown='ignore')`.
3. **Boolean Mapping:** `Late_Night_Usage` is converted from `bool` to `int` (`0`/`1`).
4. **Feature Scaling:** Continuous numerical features are standardized using `StandardScaler` fit strictly on `X_train`.
5. **Target Encoding:** `LabelEncoder` converts string labels (`Beneficial`, `Negative`, `Neutral`) into numeric IDs (`0`, `1`, `2`).

In [ ]:
# 1. Ordinal Encoding
social_order = ['Never', 'Rarely', 'Sometimes', 'Frequently', 'Always']
academic_order = ['High School', 'Undergraduate', 'Postgraduate']

social_encoder = OrdinalEncoder(categories=[social_order])
academic_encoder = OrdinalEncoder(categories=[academic_order])

X_train[['Social_Comparison_Frequency']] = social_encoder.fit_transform(X_train[['Social_Comparison_Frequency']])
X_val[['Social_Comparison_Frequency']] = social_encoder.transform(X_val[['Social_Comparison_Frequency']])
X_test[['Social_Comparison_Frequency']] = social_encoder.transform(X_test[['Social_Comparison_Frequency']])

X_train[['Academic_Level']] = academic_encoder.fit_transform(X_train[['Academic_Level']])
X_val[['Academic_Level']] = academic_encoder.transform(X_val[['Academic_Level']])
X_test[['Academic_Level']] = academic_encoder.transform(X_test[['Academic_Level']])

# 2. One-Hot Encoding for Nominal Features
nominal_cols = ['Gender', 'Primary_Platform', 'Device_Type']
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

ohe_train_df = pd.DataFrame(ohe.fit_transform(X_train[nominal_cols]), columns=ohe.get_feature_names_out(nominal_cols), index=X_train.index)
ohe_val_df = pd.DataFrame(ohe.transform(X_val[nominal_cols]), columns=ohe.get_feature_names_out(nominal_cols), index=X_val.index)
ohe_test_df = pd.DataFrame(ohe.transform(X_test[nominal_cols]), columns=ohe.get_feature_names_out(nominal_cols), index=X_test.index)

X_train = pd.concat([X_train.drop(columns=nominal_cols), ohe_train_df], axis=1)
X_val = pd.concat([X_val.drop(columns=nominal_cols), ohe_val_df], axis=1)
X_test = pd.concat([X_test.drop(columns=nominal_cols), ohe_test_df], axis=1)

# 3. Boolean Encoding
X_train['Late_Night_Usage'] = X_train['Late_Night_Usage'].astype(int)
X_val['Late_Night_Usage'] = X_val['Late_Night_Usage'].astype(int)
X_test['Late_Night_Usage'] = X_test['Late_Night_Usage'].astype(int)

# 4. Feature Scaling
num_cols = [
    'Age',
    'Daily_Usage_Hours',
    'Weekend_Extra_Hours',
    'Sleep_Duration_Hours',
    'Sleep_Quality_Score',
    'Perceived_Stress_Score',
    'Mental_Health_Index',
    'Academic_Performance_GPA'
]

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_val[num_cols] = scaler.transform(X_val[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

# 5. Target Label Encoding
label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_val_enc = label_encoder.transform(y_val)
y_test_enc = label_encoder.transform(y_test)

class_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print("Target Class Mapping:", class_mapping)
display(X_train.head())

### 7. Handling Class Imbalance with SMOTE
Because `Beneficial` (~81.8%) heavily dominates `Neutral` (~14.5%) and `Negative` (~3.7%), we apply **SMOTE** exclusively to the training set (`X_train`, `y_train_enc`) while keeping validation and test distributions untouched.

In [ ]:
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train_enc)

print("--- Class Distribution in Training Set Before SMOTE ---")
print(pd.Series(y_train_enc).map(dict(enumerate(label_encoder.classes_))).value_counts())

print("\n--- Class Distribution in Training Set After SMOTE ---")
print(pd.Series(y_train_balanced).map(dict(enumerate(label_encoder.classes_))).value_counts())

### 8. Baseline Training & Validation Comparison (4 Selected Models)
We train the 4 selected models—**Neural Network (MLP)**, **Gradient Boosting (GBM)**, **HistGradientBoosting**, and **SGD Classifier**—on `X_train_balanced` and evaluate them on `X_val` using **Accuracy**, **Macro F1-Score**, and **Weighted F1-Score**.

In [ ]:
baseline_models = {
    "Neural Network (MLP)": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42),
    "Gradient Boosting (GBM)": GradientBoostingClassifier(random_state=42),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=42),
    "SGD Classifier": SGDClassifier(random_state=42)
}

baseline_results = []

for name, model in baseline_models.items():
    model.fit(X_train_balanced, y_train_balanced)
    y_val_pred = model.predict(X_val)
    
    baseline_results.append({
        "Model Name": name,
        "Val Accuracy": round(accuracy_score(y_val_enc, y_val_pred), 4),
        "Val Macro F1": round(f1_score(y_val_enc, y_val_pred, average='macro'), 4),
        "Val Weighted F1": round(f1_score(y_val_enc, y_val_pred, average='weighted'), 4)
    })

baseline_df = pd.DataFrame(baseline_results).sort_values(by="Val Macro F1", ascending=False).reset_index(drop=True)
display(baseline_df)

### 9. Hyperparameter Tuning (Optimizing Macro F1-Score)
We tune all 4 models using `RandomizedSearchCV` (with 3-fold cross-validation optimizing `f1_macro`) and compare the tuned models against the baselines on `X_val`.

In [ ]:
param_grids = {
    "Neural Network (MLP)": {
        "hidden_layer_sizes": [(64, 32), (128, 64), (100, 50, 25)],
        "alpha": [0.0001, 0.001, 0.01],
        "learning_rate_init": [0.001, 0.005, 0.01],
        "early_stopping": [True]
    },
    "Gradient Boosting (GBM)": {
        "n_estimators": [100, 200, 300],
        "learning_rate": [0.03, 0.05, 0.1],
        "max_depth": [3, 4, 5],
        "subsample": [0.8, 1.0]
    },
    "HistGradientBoosting": {
        "max_iter": [100, 200, 300],
        "learning_rate": [0.03, 0.05, 0.1],
        "max_depth": [4, 6, None],
        "l2_regularization": [0.0, 0.1, 1.0]
    },
    "SGD Classifier": {
        "loss": ["hinge", "log_loss", "modified_huber"],
        "alpha": [0.0001, 0.001, 0.01],
        "penalty": ["l2", "elasticnet"]
    }
}

base_estimators = {
    "Neural Network (MLP)": MLPClassifier(max_iter=500, random_state=42),
    "Gradient Boosting (GBM)": GradientBoostingClassifier(random_state=42),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=42),
    "SGD Classifier": SGDClassifier(random_state=42)
}

tuned_models = {}
tuned_results = []

for name, estimator in base_estimators.items():
    print(f"Tuning {name}...")
    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=param_grids[name],
        n_iter=8,
        scoring="f1_macro",
        cv=3,
        random_state=42,
        n_jobs=-1
    )
    search.fit(X_train_balanced, y_train_balanced)
    best_est = search.best_estimator_
    tuned_models[name] = best_est
    
    y_val_pred = best_est.predict(X_val)
    tuned_results.append({
        "Model Name": name,
        "Val Accuracy": round(accuracy_score(y_val_enc, y_val_pred), 4),
        "Val Macro F1": round(f1_score(y_val_enc, y_val_pred, average='macro'), 4),
        "Val Weighted F1": round(f1_score(y_val_enc, y_val_pred, average='weighted'), 4),
        "Best Params": str(search.best_params_)
    })

tuned_df = pd.DataFrame(tuned_results).sort_values(by="Val Macro F1", ascending=False).reset_index(drop=True)
display(tuned_df)

# Plot Validation Comparison
plt.figure(figsize=(10, 4))
sns.barplot(data=tuned_df, x="Val Macro F1", y="Model Name", palette="viridis", hue="Model Name", legend=False)
plt.title("Tuned Models Comparison on Validation Set (Sorted by Macro F1-Score)")
plt.xlabel("Validation Macro F1-Score")
plt.ylabel("Model")
plt.xlim(0, 1.0)
plt.tight_layout()
plt.show()

### 10. Final Evaluation on Unseen Test Set (`X_test`)
We select the best overall model based on **Validation Macro F1-Score** (comparing both baseline and tuned models) and evaluate all 4 tuned models on `X_test`.

In [ ]:
# Pick the best model between baseline and tuned based on Validation Macro F1
best_baseline_row = baseline_df.iloc[0]
best_tuned_row = tuned_df.iloc[0]

if best_tuned_row["Val Macro F1"] >= best_baseline_row["Val Macro F1"]:
    best_model_name = best_tuned_row["Model Name"]
    best_model = tuned_models[best_model_name]
    model_version = "Tuned"
else:
    best_model_name = best_baseline_row["Model Name"]
    best_model = baseline_models[best_model_name]
    model_version = "Baseline"

print(f"==================================================")
print(f"Best Selected Model: {best_model_name} ({model_version})")
print(f"==================================================")

y_test_pred = best_model.predict(X_test)

test_acc = accuracy_score(y_test_enc, y_test_pred)
test_macro_f1 = f1_score(y_test_enc, y_test_pred, average='macro')
test_weighted_f1 = f1_score(y_test_enc, y_test_pred, average='weighted')

print(f"Test Accuracy:    {test_acc:.4f}")
print(f"Test Macro F1:    {test_macro_f1:.4f}")
print(f"Test Weighted F1: {test_weighted_f1:.4f}")

print("\n--- Detailed Classification Report (Best Model on Test Set) ---\n")
print(classification_report(y_test_enc, y_test_pred, target_names=label_encoder.classes_))

# Plot Confusion Matrix for Best Model
plt.figure(figsize=(6, 4))
cm = confusion_matrix(y_test_enc, y_test_pred)
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_
)
plt.title(f"Test Set Confusion Matrix - {best_model_name} ({model_version})")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

# Compare all 4 tuned models on Test Set
print("\n--- All 4 Tuned Models Performance on Test Set ---\n")
test_results = []
for i in range(len(tuned_df)):
    m_name = tuned_df.loc[i, "Model Name"]
    m = tuned_models[m_name]
    preds = m.predict(X_test)
    test_results.append({
        "Rank": i + 1,
        "Model Name": m_name,
        "Test Accuracy": round(accuracy_score(y_test_enc, preds), 4),
        "Test Macro F1": round(f1_score(y_test_enc, preds, average='macro'), 4),
        "Test Weighted F1": round(f1_score(y_test_enc, preds, average='weighted'), 4)
    })

display(pd.DataFrame(test_results))

### 11. Model Explainability: Feature Importance
Using model-agnostic **Permutation Importance** on `X_test`, we identify which features contribute most to predicting `Overall_Impact`.

In [ ]:
perm_importance = permutation_importance(
    best_model, X_test, y_test_enc, n_repeats=10, scoring="f1_macro", random_state=42, n_jobs=-1
)

importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance Mean": perm_importance.importances_mean,
    "Importance Std": perm_importance.importances_std
}).sort_values(by="Importance Mean", ascending=False).head(12)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df, x="Importance Mean", y="Feature", palette="magma", hue="Feature", legend=False)
plt.title(f"Top 12 Most Important Features ({best_model_name})")
plt.xlabel("Mean Decrease in Macro F1-Score")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

display(importance_df)

### 12. Export Model & Preprocessing Pipeline for Web App
We bundle `best_model` together with all fitted preprocessors (`stress_imputer`, `gpa_imputer`, `social_encoder`, `academic_encoder`, `ohe`, `scaler`, `label_encoder`) into `social_media_impact_pipeline.joblib` so it can be directly loaded into a Streamlit, Gradio, or FastAPI web application.

In [ ]:
deployment_bundle = {
    "model_name": f"{best_model_name} ({model_version})",
    "model": best_model,
    "stress_imputer": stress_imputer,
    "gpa_imputer": gpa_imputer,
    "social_encoder": social_encoder,
    "academic_encoder": academic_encoder,
    "ohe": ohe,
    "scaler": scaler,
    "label_encoder": label_encoder,
    "nominal_cols": nominal_cols,
    "num_cols": num_cols,
    "feature_columns": list(X_train.columns)
}

bundle_filename = "social_media_impact_pipeline.joblib"
joblib.dump(deployment_bundle, bundle_filename)
print(f"Saved deployment bundle successfully to '{bundle_filename}'")

# Quick sanity check: reload and predict on 1 test sample
loaded_bundle = joblib.load(bundle_filename)
sample_pred_idx = loaded_bundle["model"].predict(X_test.iloc[:1])[0]
sample_pred_label = loaded_bundle["label_encoder"].inverse_transform([sample_pred_idx])[0]
print("Sanity Check Prediction on 1st Test Sample:", sample_pred_label)